# Data Cleaning Phase
This notebook performs data cleaning on the healthcare analytics datasets.

In [1]:
import pandas as pd
import numpy as np
import os

os.makedirs('../data/cleaned', exist_ok=True)

## 1. Load Data

In [2]:
appointments = pd.read_csv('../data/raw/appointments.csv')
billing = pd.read_csv('../data/raw/billing.csv')
doctors = pd.read_csv('../data/raw/doctors.csv')
patients = pd.read_csv('../data/raw/patients.csv')
treatments = pd.read_csv('../data/raw/treatments.csv')

## 2. Inspect & Clean: Patients Dataset

In [3]:
print('Shape:', patients.shape)
print('\nColumns:', patients.columns.tolist())
print('\nData Types:\n', patients.dtypes)
display(patients.head())

Shape: (50, 11)

Columns: ['patient_id', 'first_name', 'last_name', 'gender', 'date_of_birth', 'contact_number', 'address', 'registration_date', 'insurance_provider', 'insurance_number', 'email']

Data Types:
 patient_id            object
first_name            object
last_name             object
gender                object
date_of_birth         object
contact_number         int64
address               object
registration_date     object
insurance_provider    object
insurance_number      object
email                 object
dtype: object


,patient_id,first_name,last_name,gender,date_of_birth,contact_number,address,registration_date,insurance_provider,insurance_number,email
0,P001,David,Williams,F,1955-06-04,6939585183,789 Pine Rd,2022-06-23,WellnessCorp,INS840674,david.williams@mail.com
1,P002,Emily,Smith,F,1984-10-12,8228188767,321 Maple Dr,2022-01-15,PulseSecure,INS354079,emily.smith@mail.com
2,P003,Laura,Jones,M,1977-08-21,8397029847,321 Maple Dr,2022-02-07,PulseSecure,INS650929,laura.jones@mail.com
3,P004,Michael,Johnson,F,1981-02-20,9019443432,123 Elm St,2021-03-02,HealthIndia,INS789944,michael.johnson@mail.com
4,P005,David,Wilson,M,1960-06-23,7734463155,123 Elm St,2021-09-29,MedCare Plus,INS788105,david.wilson@mail.com


In [4]:
print('Missing Values:\n', patients.isnull().sum())
print('\nDuplicates:', patients.duplicated().sum())

Missing Values:
 patient_id            0
first_name            0
last_name             0
gender                0
date_of_birth         0
contact_number        0
address               0
registration_date     0
insurance_provider    0
insurance_number      0
email                 0
dtype: int64

Duplicates: 0


In [5]:
# Cleaning Patients
# Reason: Dropping exact duplicates ensures we don't double-count patients.
patients_clean = patients.copy()
patients_clean.drop_duplicates(inplace=True)

# Reason: Some patients might not have insurance or email. Using placeholder values preserves the rest of their important demographic info.
patients_clean['insurance_provider'].fillna('No Insurance', inplace=True)
patients_clean['insurance_number'].fillna('None', inplace=True)
patients_clean['email'].fillna('Unknown', inplace=True)

# Standardize text
patients_clean['gender'] = patients_clean['gender'].str.upper().str.strip()

# Correct types
patients_clean['date_of_birth'] = pd.to_datetime(patients_clean['date_of_birth'], errors='coerce')
patients_clean['registration_date'] = pd.to_datetime(patients_clean['registration_date'], errors='coerce')

# Validation
print('Cleaned Shape:', patients_clean.shape)
print('Remaining Missing:\n', patients_clean.isnull().sum())

Cleaned Shape: (50, 11)
Remaining Missing:
 patient_id            0
first_name            0
last_name             0
gender                0
date_of_birth         0
contact_number        0
address               0
registration_date     0
insurance_provider    0
insurance_number      0
email                 0
dtype: int64


C:\Users\DELL\AppData\Local\Temp\ipykernel_2192\250827435.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  patients_clean['insurance_provider'].fillna('No Insurance', inplace=True)
C:\Users\DELL\AppData\Local\Temp\ipykernel_2192\250827435.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves 

## 3. Inspect & Clean: Doctors Dataset

In [6]:
print('Shape:', doctors.shape)
print('\nColumns:', doctors.columns.tolist())
print('\nData Types:\n', doctors.dtypes)
display(doctors.head())

Shape: (10, 8)

Columns: ['doctor_id', 'first_name', 'last_name', 'specialization', 'phone_number', 'years_experience', 'hospital_branch', 'email']

Data Types:
 doctor_id           object
first_name          object
last_name           object
specialization      object
phone_number         int64
years_experience     int64
hospital_branch     object
email               object
dtype: object


,doctor_id,first_name,last_name,specialization,phone_number,years_experience,hospital_branch,email
0,D001,David,Taylor,Dermatology,8322010158,17,Westside Clinic,dr.david.taylor@hospital.com
1,D002,Jane,Davis,Pediatrics,9004382050,24,Eastside Clinic,dr.jane.davis@hospital.com
2,D003,Jane,Smith,Pediatrics,8737740598,19,Eastside Clinic,dr.jane.smith@hospital.com
3,D004,David,Jones,Pediatrics,6594221991,28,Central Hospital,dr.david.jones@hospital.com
4,D005,Sarah,Taylor,Dermatology,9118538547,26,Central Hospital,dr.sarah.taylor@hospital.com


In [7]:
print('Missing Values:\n', doctors.isnull().sum())
print('\nDuplicates:', doctors.duplicated().sum())

Missing Values:
 doctor_id           0
first_name          0
last_name           0
specialization      0
phone_number        0
years_experience    0
hospital_branch     0
email               0
dtype: int64

Duplicates: 0


In [8]:
# Cleaning Doctors
doctors_clean = doctors.copy()
doctors_clean.drop_duplicates(inplace=True)

# Standardize text
doctors_clean['specialization'] = doctors_clean['specialization'].str.strip().str.title()

# Check outliers in experience
# Reason: Negative years of experience is an invalid data entry. Replacing with the median preserves the record while assuming typical experience.
if (doctors_clean['years_experience'] < 0).any():
    med_exp = doctors_clean.loc[doctors_clean['years_experience'] >= 0, 'years_experience'].median()
    doctors_clean.loc[doctors_clean['years_experience'] < 0, 'years_experience'] = med_exp

# Validation
print('Cleaned Shape:', doctors_clean.shape)

Cleaned Shape: (10, 8)


## 4. Inspect & Clean: Appointments Dataset

In [9]:
print('Shape:', appointments.shape)
print('\nColumns:', appointments.columns.tolist())
print('\nData Types:\n', appointments.dtypes)
display(appointments.head())

Shape: (200, 7)

Columns: ['appointment_id', 'patient_id', 'doctor_id', 'appointment_date', 'appointment_time', 'reason_for_visit', 'status']

Data Types:
 appointment_id      object
patient_id          object
doctor_id           object
appointment_date    object
appointment_time    object
reason_for_visit    object
status              object
dtype: object


,appointment_id,patient_id,doctor_id,appointment_date,appointment_time,reason_for_visit,status
0,A001,P034,D009,2023-08-09,15:15:00,Therapy,Scheduled
1,A002,P032,D004,2023-06-09,14:30:00,Therapy,No-show
2,A003,P048,D004,2023-06-28,8:00:00,Consultation,Cancelled
3,A004,P025,D006,2023-09-01,9:15:00,Consultation,Cancelled
4,A005,P040,D003,2023-07-06,12:45:00,Emergency,No-show


In [10]:
print('Missing Values:\n', appointments.isnull().sum())
print('\nDuplicates:', appointments.duplicated().sum())

Missing Values:
 appointment_id      0
patient_id          0
doctor_id           0
appointment_date    0
appointment_time    0
reason_for_visit    0
status              0
dtype: int64

Duplicates: 0


In [11]:
# Cleaning Appointments
appointments_clean = appointments.copy()
appointments_clean.drop_duplicates(inplace=True)

# Handle missing
# Reason: A missing reason for visit shouldn't invalidate the appointment record itself.
appointments_clean['reason_for_visit'].fillna('Unknown', inplace=True)
appointments_clean['status'] = appointments_clean['status'].str.strip().str.title()
appointments_clean['status'].fillna('Unknown', inplace=True)

# Types
appointments_clean['appointment_date'] = pd.to_datetime(appointments_clean['appointment_date'], errors='coerce')
# For simplicity, keeping appointment_time as string, or can convert to datetime time
appointments_clean['appointment_time'] = pd.to_datetime(appointments_clean['appointment_time'], format='%H:%M:%S', errors='coerce').dt.time

print('Cleaned Shape:', appointments_clean.shape)

Cleaned Shape:

 (200, 7)


C:\Users\DELL\AppData\Local\Temp\ipykernel_2192\2019803739.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  appointments_clean['reason_for_visit'].fillna('Unknown', inplace=True)
C:\Users\DELL\AppData\Local\Temp\ipykernel_2192\2019803739.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves a

## 5. Inspect & Clean: Treatments Dataset

In [12]:
print('Shape:', treatments.shape)
print('\nColumns:', treatments.columns.tolist())
print('\nData Types:\n', treatments.dtypes)
display(treatments.head())

Shape: (200, 6)

Columns: ['treatment_id', 'appointment_id', 'treatment_type', 'description', 'cost', 'treatment_date']

Data Types:
 treatment_id       object
appointment_id     object
treatment_type     object
description        object
cost              float64
treatment_date     object
dtype: object


,treatment_id,appointment_id,treatment_type,description,cost,treatment_date
0,T001,A001,Chemotherapy,Basic screening,3941.97,2023-08-09
1,T002,A002,MRI,Advanced protocol,4158.44,2023-06-09
2,T003,A003,MRI,Standard procedure,3731.55,2023-06-28
3,T004,A004,MRI,Basic screening,4799.86,2023-09-01
4,T005,A005,ECG,Standard procedure,582.05,2023-07-06


In [13]:
print('Missing Values:\n', treatments.isnull().sum())
print('\nDuplicates:', treatments.duplicated().sum())

Missing Values:
 treatment_id      0
appointment_id    0
treatment_type    0
description       0
cost              0
treatment_date    0
dtype: int64

Duplicates: 0


In [14]:
# Cleaning Treatments
treatments_clean = treatments.copy()
treatments_clean.drop_duplicates(inplace=True)

# Types & Missing
treatments_clean['treatment_date'] = pd.to_datetime(treatments_clean['treatment_date'], errors='coerce')
treatments_clean['cost'] = pd.to_numeric(treatments_clean['cost'], errors='coerce')

# Reason: Replacing missing or negative cost with median avoids dropping valuable treatment history while estimating a standard cost.
treatments_clean['cost'].fillna(treatments_clean['cost'].median(), inplace=True)
treatments_clean.loc[treatments_clean['cost'] < 0, 'cost'] = treatments_clean['cost'].median()

print('Cleaned Shape:', treatments_clean.shape)

Cleaned Shape: (200, 6)


C:\Users\DELL\AppData\Local\Temp\ipykernel_2192\2759383094.py:10: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  treatments_clean['cost'].fillna(treatments_clean['cost'].median(), inplace=True)


## 6. Inspect & Clean: Billing Dataset

In [15]:
print('Shape:', billing.shape)
print('\nColumns:', billing.columns.tolist())
print('\nData Types:\n', billing.dtypes)
display(billing.head())

Shape: (200, 7)

Columns: ['bill_id', 'patient_id', 'treatment_id', 'bill_date', 'amount', 'payment_method', 'payment_status']

Data Types:
 bill_id            object
patient_id         object
treatment_id       object
bill_date          object
amount            float64
payment_method     object
payment_status     object
dtype: object


,bill_id,patient_id,treatment_id,bill_date,amount,payment_method,payment_status
0,B001,P034,T001,2023-08-09,3941.97,Insurance,Pending
1,B002,P032,T002,2023-06-09,4158.44,Insurance,Paid
2,B003,P048,T003,2023-06-28,3731.55,Insurance,Paid
3,B004,P025,T004,2023-09-01,4799.86,Insurance,Failed
4,B005,P040,T005,2023-07-06,582.05,Credit Card,Pending


In [16]:
print('Missing Values:\n', billing.isnull().sum())
print('\nDuplicates:', billing.duplicated().sum())

Missing Values:
 bill_id           0
patient_id        0
treatment_id      0
bill_date         0
amount            0
payment_method    0
payment_status    0
dtype: int64

Duplicates: 0


In [17]:
# Cleaning Billing
billing_clean = billing.copy()
billing_clean.drop_duplicates(inplace=True)

# Types & Missing
billing_clean['bill_date'] = pd.to_datetime(billing_clean['bill_date'], errors='coerce')
billing_clean['amount'] = pd.to_numeric(billing_clean['amount'], errors='coerce')

# Reason: Replacing negative or missing amounts with median to keep billing records intact.
billing_clean['amount'].fillna(billing_clean['amount'].median(), inplace=True)
billing_clean.loc[billing_clean['amount'] < 0, 'amount'] = billing_clean['amount'].median()

print('Cleaned Shape:', billing_clean.shape)

Cleaned Shape: (200, 7)


C:\Users\DELL\AppData\Local\Temp\ipykernel_2192\1711749710.py:10: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  billing_clean['amount'].fillna(billing_clean['amount'].median(), inplace=True)


## 7. Save Cleaned Datasets

In [18]:
patients_clean.to_csv('../data/cleaned/patients.csv', index=False)
doctors_clean.to_csv('../data/cleaned/doctors.csv', index=False)
appointments_clean.to_csv('../data/cleaned/appointments.csv', index=False)
treatments_clean.to_csv('../data/cleaned/treatments.csv', index=False)
billing_clean.to_csv('../data/cleaned/billing.csv', index=False)
print('All cleaned datasets saved successfully.')

All cleaned datasets saved successfully.


## Data Quality Summary

In [19]:
print('--- Data Quality Summary ---')
print(f'Patients: Original={len(patients)}, Cleaned={len(patients_clean)}, Missing Handled={patients.isnull().sum().sum()}, Duplicates Handled={patients.duplicated().sum()}')
print(f'Doctors: Original={len(doctors)}, Cleaned={len(doctors_clean)}, Missing Handled={doctors.isnull().sum().sum()}, Duplicates Handled={doctors.duplicated().sum()}')
print(f'Appointments: Original={len(appointments)}, Cleaned={len(appointments_clean)}, Missing Handled={appointments.isnull().sum().sum()}, Duplicates Handled={appointments.duplicated().sum()}')
print(f'Treatments: Original={len(treatments)}, Cleaned={len(treatments_clean)}, Missing Handled={treatments.isnull().sum().sum()}, Duplicates Handled={treatments.duplicated().sum()}')
print(f'Billing: Original={len(billing)}, Cleaned={len(billing_clean)}, Missing Handled={billing.isnull().sum().sum()}, Duplicates Handled={billing.duplicated().sum()}')
print('\nKey Transformations:')
print('- Converted string dates to proper datetime objects.')
print('- Checked for and replaced negative values/outliers (e.g. costs, experience) with median values.')
print('- Filled missing categorical text with descriptive placeholders (Unknown/No Insurance) to preserve records.')
print('- Removed exact duplicate rows from all datasets.')

--- Data Quality Summary ---
Patients: Original=50, Cleaned=50, Missing Handled=0, Duplicates Handled=0
Doctors: Original=10, Cleaned=10, Missing Handled=0, Duplicates Handled=0
Appointments: Original=200, Cleaned=200, Missing Handled=0, Duplicates Handled=0
Treatments: Original=200, Cleaned=200, Missing Handled=0, Duplicates Handled=0
Billing: Original=200, Cleaned=200, Missing Handled=0, Duplicates Handled=0

Key Transformations:
- Converted string dates to proper datetime objects.
- Checked for and replaced negative values/outliers (e.g. costs, experience) with median values.
- Filled missing categorical text with descriptive placeholders (Unknown/No Insurance) to preserve records.
- Removed exact duplicate rows from all datasets.
